# Social Media Sentiment Analysis — Multi-Class NLP Classification

**End-to-end Machine Learning portfolio: regex text preprocessing, TF-IDF n-grams feature extraction, imbalanced class handling, linear classification, model explainability, and Streamlit prototype.**

Proyek ini bertujuan mengklasifikasikan sentimen opini pelanggan penerbangan di media sosial (Twitter/X) ke dalam 3 kategori: **Negative**, **Neutral**, dan **Positive**.

## Ringkasan Metodologi & Hasil
- **Dataset:** 14.640 data tweet nyata dari Kaggle (*Twitter US Airline Sentiment*).
- **Distribusi Target:** Imbalanced class (Negative 62,69%, Neutral 21,17%, Positive 16,14%).
- **Text Cleaning:** Regex pipeline untuk menghapus @mentions, URL, entitas HTML, tanda baca, serta normalisasi lowercase.
- **Ekstraksi Fitur:** TF-IDF Vectorizer dengan unigram & bigram (`ngram_range=(1, 2)`), sublinear TF scaling, dan 5.000 fitur kata paling informatif.
- **Model Terbaik:** **Logistic Regression (`class_weight='balanced'`, $C=0.5$)** dengan metrik optimasi **Macro F1-Score**.
- **Evaluasi Akhir Data Uji (Holdout Test Set — 2.928 Tweet):**
  - **Akurasi:** `74,62%`
  - **Macro F1-Score:** `0,7058`
  - **Weighted F1-Score:** `0,7557`
  - **F1-Score Negative:** `0,8242` (Precision 89,31%)


In [ ]:
import sys
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

print('Libraries loaded successfully!')


## 1. Memuat Dataset

Loader berikut mendeteksi secara otomatis lingkungan kerja:
- **Kaggle:** Mencari file di `/kaggle/input/twitter-airline-sentiment/Tweets.csv`
- **Lokal:** Membaca dari `data/Tweets.csv` atau direktori saat ini
- **Google Colab:** Meminta upload jika file belum ada


In [ ]:
file_name = 'Tweets.csv'
data_path = None

candidates = [
    Path(file_name),
    Path('data') / file_name,
    Path('../data') / file_name,
    Path('../Tweets.csv')
]

for p in candidates:
    if p.is_file():
        data_path = p
        break

if data_path is None and Path('/kaggle/input').is_dir():
    matches = sorted(Path('/kaggle/input').rglob(file_name))
    if matches:
        data_path = matches[0]

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = Path(file_name)
    except ImportError:
        raise FileNotFoundError('File Tweets.csv tidak ditemukan. Letakkan di folder data/.')

print('Memuat dataset dari:', data_path)
df = pd.read_csv(data_path)
df[['text', 'airline_sentiment', 'airline']].head()


## 2. Pemeriksaan Struktur Data & Distribusi Kelas Target

Memeriksa kelengkapan baris dan rasio ketimpangan kelas (*class imbalance*).


In [ ]:
print(f'Total baris tweet : {df.shape[0]:,}')
print(f'Total kolom       : {df.shape[1]}')

sentiment_counts = df['airline_sentiment'].value_counts()
sentiment_pct = (df['airline_sentiment'].value_counts(normalize=True) * 100).round(2)

dist_df = pd.DataFrame({
    'Jumlah Tweet': sentiment_counts,
    'Persentase (%)': sentiment_pct
})

print('\n--- DISTRIBUSI KELAS SENTIMEN ---')
print(dist_df)
print(f'\nMissing pada teks  : {df["text"].isnull().sum()}')
print(f'Missing pada label : {df["airline_sentiment"].isnull().sum()}')


## 3. Pembersihan Teks Media Sosial (*Text Preprocessing*)

Menghilangkan *noise* khas media sosial:
- Mengubah ke huruf kecil (*lowercase*)
- Menghapus entitas HTML (`&amp;`, dll.)
- Menghapus URL / tautan web
- Menghapus mention akun (`@username`)
- Menghapus angka dan karakter non-alfabet
- Menghapus spasi berlebih


In [ ]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'#[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_tweet)

print('Contoh Pembersihan Teks:')
for i in [3, 4, 1]:
    print(f'\n[Asli]    : {df["text"].iloc[i]}')
    print(f'[Cleaned] : {df["cleaned_text"].iloc[i]}')


## 4. Analisis Sentimen per Maskapai Penerbangan

Memvisualisasikan bagaimana sentimen pelanggan bervariasi antarmaskapai.


In [ ]:
plt.figure(figsize=(12, 5))
palette = {'negative': '#E76F51', 'neutral': '#E9C46A', 'positive': '#2A9D8F'}

sns.countplot(
    data=df, 
    x='airline', 
    hue='airline_sentiment', 
    palette=palette,
    order=df['airline'].value_counts().index
)

plt.title('Distribusi Sentimen per Maskapai Penerbangan', fontsize=14, pad=15)
plt.xlabel('Maskapai', fontsize=12)
plt.ylabel('Jumlah Tweet', fontsize=12)
plt.legend(title='Sentimen')
sns.despine()
plt.tight_layout()
plt.show()


## 5. Pembagian Data Latih/Uji & Vektorisasi TF-IDF

Membagi data dengan stratifikasi (80% Train, 20% Holdout Test), lalu mengubah teks menjadi representasi numerik menggunakan **TF-IDF (Unigram + Bigram, 5.000 fitur)**.


In [ ]:
X = df['cleaned_text']
y = df['airline_sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train: {len(X_train):,} tweet | Test: {len(X_test):,} tweet')

tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Bentuk Matriks TF-IDF Latih: {X_train_tfidf.shape}')


## 6. Perbandingan Model (5-Fold Stratified Cross Validation)

Membandingkan Dummy Classifier, Multinomial Naive Bayes, Logistic Regression (Class-Weighted), dan Linear SVM.


In [ ]:
models = {
    'Dummy (Majority Baseline)': DummyClassifier(strategy='most_frequent'),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.5),
    'Logistic Regression (Balanced)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Linear SVM (LinearSVC Balanced)': LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'Macro_F1': 'f1_macro', 'Accuracy': 'accuracy', 'Weighted_F1': 'f1_weighted'}
cv_results = []

for name, clf in models.items():
    pipe = Pipeline([('tfidf', tfidf), ('clf', clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    cv_results.append({
        'Model': name,
        'Macro F1-Score': scores['test_Macro_F1'].mean(),
        'Accuracy (%)': scores['test_Accuracy'].mean() * 100,
        'Weighted F1 (%)': scores['test_Weighted_F1'].mean() * 100
    })

pd.DataFrame(cv_results).sort_values('Macro F1-Score', ascending=False).reset_index(drop=True)


## 7. Hyperparameter Tuning & Evaluasi Akhir Data Uji

Melakukan tuning regularisasi $C$ pada Logistic Regression dan mengevaluasi 2.928 tweet data uji independen.


In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

grid_search = GridSearchCV(
    pipeline, {'clf__C': [0.5, 1.0, 2.0, 3.0]}, cv=skf, scoring='f1_macro', n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f'Nilai C Terbaik         : {grid_search.best_params_["clf__C"]}')
print(f'Best CV Macro F1-Score : {grid_search.best_score_:.4f}')

y_pred = best_model.predict(X_test)

print('\n=== LAPORAN EVALUASI AKHIR (DATA UJI) ===')
print(classification_report(y_test, y_pred, digits=4))

fig, ax = plt.subplots(figsize=(6, 5))
labels = ['negative', 'neutral', 'positive']

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=labels, display_labels=['Negative', 'Neutral', 'Positive'],
    cmap='Blues', values_format='d', ax=ax
)
plt.title('Confusion Matrix — Data Uji', fontsize=13, pad=12)
plt.tight_layout()
plt.show()


## 8. Analisis Kata Kunci Paling Berpengaruh (*Model Explainability*)

Memeriksa fitur kata yang memiliki koefisien bobot tertinggi untuk masing-masing kelas sentimen.


In [ ]:
vectorizer = best_model.named_steps['tfidf']
classifier = best_model.named_steps['clf']
feature_names = np.array(vectorizer.get_feature_names_out())
classes = classifier.classes_

for i, class_label in enumerate(classes):
    top10_idx = np.argsort(classifier.coef_[i])[-10:]
    top_words = feature_names[top10_idx][::-1]
    top_weights = classifier.coef_[i][top10_idx][::-1]
    print(f'\n--- Sentimen: [{class_label.upper()}] ---')
    for word, weight in zip(top_words, top_weights):
        print(f'  {word:<15} : +{weight:.3f}')


## 9. Menyimpan Artefak Model untuk Deployment Aplikasi

Menyimpan pipeline utuh ke format joblib beserta metadata dan gambar evaluasi.


In [ ]:
out_dir = Path('sentiment_artifacts')
out_dir.mkdir(exist_ok=True)

# Latih ulang pada seluruh dataset
final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf', LogisticRegression(C=0.5, class_weight='balanced', random_state=42, max_iter=1000))
])
final_pipeline.fit(df['cleaned_text'], df['airline_sentiment'])

bundle = {
    'pipeline': final_pipeline,
    'classes': final_pipeline.classes_.tolist(),
    'top_words_per_class': {
        cls: [feature_names[idx] for idx in np.argsort(classifier.coef_[i])[-15:][::-1]]
        for i, cls in enumerate(classes)
    }
}
joblib.dump(bundle, out_dir / 'sentiment_model.joblib')
fig.savefig(out_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')

print('Pipeline sentimen berhasil disimpan ke sentiment_artifacts/!')
